In [1]:
from pathlib import Path

from lassi.profiler import Timer, MultiProfiler, GPUProfiler, CPUProfiler
from lassi.source_file import SourceFile
from lassi.executer import FunctionalValidator
from typing import Annotated

from openai import AsyncOpenAI
from groq import Groq

from lassi.compiler import Compiler, CompilerTool, CompilationError

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

import subprocess

## Minimal C Example

In [ ]:
# Declare a new file

source_file = SourceFile(
    file_name = Path("test.c")
)

source_file

In [ ]:
source_file.write( # this could be the LLM output
    """#include <stdio.h>

        int main(void) {
            printf("Compiler test successful!");
            return 0;
        }
    """
)

executable = source_file.compile()
success = source_file.execute()
report = source_file.get_last_execution_report()

In [ ]:
report # default report is latency

## More complex CUDA example

In [ ]:
source_file = SourceFile(
    file_name = Path("similarity_cuda_test.cu"),
    folder_path = Path("/home/gbrun/test_cuda_folder/src/") # <-- from another folder
)

In [ ]:
source_file # this time lang is CUDA

In [ ]:
source_file.compile(
    kwds="-O3", # all flags go here
    output_file=Path("test_cuda") # custom name
    ) 

In [ ]:
source_file.execute(
    args="100 100", # need args
    profiler=MultiProfiler([Timer(),CPUProfiler(),GPUProfiler()]) # GPU profiler. There is NVIDIA and AMD
    )

In [ ]:
source_file.get_last_execution_report() # more complex report

## Code Generation

In [ ]:
import dotenv
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

from openai import AsyncOpenAI

In [ ]:
dotenv.load_dotenv()

In [ ]:
source_file = SourceFile(
    file_name = Path("llm_generated_code.c"),
)

In [ ]:
@dataclass
class CodeGenDependencies:
    language: Annotated[str, Field(description="Language to implement the code in.")]

@dataclass
class CodeGenOutput:
    code: Annotated[str, Field(description="Valid code that follows the requested task. Code only.")]

In [ ]:
MODEL_NAME = "openai/gpt-oss-120b"

### Groq model

In [ ]:
model = GroqModel(MODEL_NAME)

### Locally hosted OpenAI (vLLM)

In [ ]:
# model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(base_url="http://localhost:8000/v1"))

### Declare Agent

In [ ]:
agent = Agent(
    model=model,
    instructions=(
        "You are a helpful coding tool."
    ),
    output_type= CodeGenOutput,
)

### Create and test a new file

In [ ]:
deps = CodeGenDependencies(language=source_file.lang.value)

result = await agent.run("Write me a C program that takes from args a number N and prints the first N args of the fibonacci sequence.")

source_file.write(result.output.code)

In [ ]:
print(source_file.read())

In [ ]:
source_file.compile()

unit_test = [
    FunctionalValidator(
        args = "4",
        golden_output="0 1 1 2\n",
        ret_code=0),
    FunctionalValidator(
        args = "1",
        golden_output="0\n",
        ret_code=0),
    FunctionalValidator(
        args = "5",
        golden_output="0 1 1 2 3\n",
        ret_code=0),
]

output = source_file.execute(
    validator = unit_test,
)

source_file.get_execution_history()

In [ ]:
source_file.compile()

output = source_file.execute(
    args = "10000000",
    profiler=CPUProfiler()
)

source_file.get_execution_history()

In [ ]:
cpuprof = CPUProfiler()
cpuprof.start()

In [ ]:
cpuprof.stop()

In [ ]:
list(cpuprof._probe.powers)

In [ ]:
output

In [ ]:
source_file

## PolyBench

In [ ]:
class CustomCompilerTool(CompilerTool):
    def compile(
        self,
        file: Path = None,
        output_file : Path = None,
        kwds: str = None,
    ) -> Path:

        # Build command
        cmd = [
            "gcc",
            "-O3",
            "-I", "/home/gbrun/PolyBenchC-4.2.1/utilities",
            "-I", "/home/gbrun/PolyBenchC-4.2.1/linear-algebra/kernels/atax",
            "/home/gbrun/PolyBenchC-4.2.1/utilities/polybench.c",
            "/home/gbrun/PolyBenchC-4.2.1/linear-algebra/kernels/atax/atax.c",
            "-DPOLYBENCH_TIME",
            "-DEXTRALARGE_DATASET",
            "-o", str(output_file),
        ]

        print(f"Compiling with command: {' '.join(cmd)}")

        # Run compiler
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            raise CompilationError(f"Compilation failed:\n{result.stderr}")

        return output_file

In [ ]:

souce_file = source_file = SourceFile(
    file_name = Path("atax.c"),
    folder_path=Path("/home/gbrun/PolyBenchC-4.2.1/linear-algebra/kernels/atax/"),
    compiler_tool=CustomCompilerTool(
        compiler=Compiler.GCC),
)

souce_file

In [ ]:
souce_file.compile(
    output_file = Path("/home/gbrun/PolyBenchC-4.2.1/linear-algebra/my_bench")
)